# 06 — Bag NMS Ensemble → Submission

Runs all 7 bag `best.pt` models over the 786 test images, merges per-image boxes
with NMS (iou_threshold=0.25), and writes the final COCO JSON for submission.

Same NMS recipe that scored 0.4430 on the k-fold ensemble.

## Paths & config

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR_WBF")
RUNS_DIR     = PROJECT_ROOT / "runs/kfold"
TEST_IMG_DIR = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR/ClearSAR/data/images/test")
OUT_JSON     = PROJECT_ROOT / "bag_nms_ensemble_preds.json"

N_BAGS       = 7
NMS_IOU_THR  = 0.25
CONF_THR     = 0.01   # low threshold — NMS + COCO eval will sort it out

BAG_WEIGHTS = [
    RUNS_DIR / f"bag_{i}_rtdetr_x/weights/best.pt"
    for i in range(N_BAGS)
]

for p in BAG_WEIGHTS:
    assert p.is_file(), f"missing {p}"
    print("OK:", p.parent.parent.name)

print(f"\nTest images: {len(list(TEST_IMG_DIR.iterdir()))}")

OK: bag_0_rtdetr_x
OK: bag_1_rtdetr_x
OK: bag_2_rtdetr_x
OK: bag_3_rtdetr_x
OK: bag_4_rtdetr_x
OK: bag_5_rtdetr_x
OK: bag_6_rtdetr_x

Test images: 786


## Per-bag OOF sanity check

Quick read of each bag's best val mAP50-95 from its own results.csv.

In [ ]:
import pandas as pd

for i in range(N_BAGS):
    csv = RUNS_DIR / f"bag_{i}_rtdetr_x/results.csv"
    df  = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    col = next(c for c in df.columns if "mAP50-95" in c)
    best = df[col].max()
    best_ep = int(df.loc[df[col].idxmax(), "epoch"])
    print(f"bag_{i}: best OOF mAP50-95 = {best:.4f}  (epoch {best_ep})")

bag_0: best OOF mAP50-95 = 0.3821  (epoch 46)
bag_1: best OOF mAP50-95 = 0.4136  (epoch 80)
bag_2: best OOF mAP50-95 = 0.4361  (epoch 77)
bag_3: best OOF mAP50-95 = 0.4642  (epoch 66)
bag_4: best OOF mAP50-95 = 0.4284  (epoch 74)
bag_5: best OOF mAP50-95 = 0.3998  (epoch 77)
bag_6: best OOF mAP50-95 = 0.4257  (epoch 70)


## Inference — collect per-image boxes from all 7 bags

One model loaded at a time to keep VRAM clean.
Stores raw boxes in `all_preds`: `{image_id: [{xyxy, score, cls}, ...]}`.

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import torch, numpy as np

# all_preds[image_id] = list of (x1,y1,x2,y2,score,cls) across all bags
all_preds = defaultdict(list)

for i, pt in enumerate(BAG_WEIGHTS):
    print(f"\n--- bag_{i} inference ---")
    model = YOLO(str(pt))
    for r in model.predict(
        source=str(TEST_IMG_DIR),
        conf=CONF_THR,
        imgsz=640,
        device=0,
        save=False,
        verbose=False,
        stream=True,
    ):
        image_id = int(Path(r.path).stem)
        xyxy  = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clses = r.boxes.cls.cpu().numpy().astype(int)
        for box, score, cls in zip(xyxy, confs, clses):
            all_preds[image_id].append((*box.tolist(), float(score), int(cls)))

    del model
    torch.cuda.empty_cache()
    n_dets = sum(len(v) for v in all_preds.values())
    print(f"bag_{i} done — running total detections: {n_dets}")

print(f"\nAll bags done. Images with detections: {len(all_preds)}")


--- bag_0 inference ---
bag_0 done — running total detections: 235800

--- bag_1 inference ---
bag_1 done — running total detections: 471600

--- bag_2 inference ---
bag_2 done — running total detections: 707400

--- bag_3 inference ---
bag_3 done — running total detections: 943200

--- bag_4 inference ---
bag_4 done — running total detections: 1179000

--- bag_5 inference ---
bag_5 done — running total detections: 1414800

--- bag_6 inference ---
bag_6 done — running total detections: 1650600

All bags done. Images with detections: 786


## NMS merge

`iou_threshold=0.25` — same value that beat WBF on the k-fold ensemble.

In [ ]:
import torchvision
import json

coco_preds = []

for image_id, dets in all_preds.items():
    boxes  = torch.tensor([[d[0],d[1],d[2],d[3]] for d in dets], dtype=torch.float32)
    scores = torch.tensor([d[4] for d in dets],                   dtype=torch.float32)
    clses  = [d[5] for d in dets]

    keep = torchvision.ops.nms(boxes, scores, iou_threshold=NMS_IOU_THR)

    for k in keep:
        x1, y1, x2, y2 = boxes[k].tolist()
        coco_preds.append({
            "image_id":    image_id,
            "category_id": clses[k] + 1,
            "bbox":  [x1, y1, x2 - x1, y2 - y1],
            "score": float(scores[k]),
        })

print(f"Before NMS: {sum(len(v) for v in all_preds.values()):,} detections")
print(f"After  NMS: {len(coco_preds):,} detections")
print(f"Reduction:  {100*(1 - len(coco_preds)/sum(len(v) for v in all_preds.values())):.1f}%")

Before NMS: 1,650,600 detections
After  NMS: 204,586 detections
Reduction:  87.6%


## Write submission JSON

In [ ]:
OUT_JSON.write_text(json.dumps(coco_preds, indent=2))
print(f"Written: {OUT_JSON}")
print(f"Total detections in submission: {len(coco_preds):,}")

# Quick sanity: score distribution
scores_all = [d["score"] for d in coco_preds]
print(f"Score — min: {min(scores_all):.3f}  median: {float(np.median(scores_all)):.3f}  max: {max(scores_all):.3f}")

Written: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\bag_nms_ensemble_preds.json
Total detections in submission: 204,586
Score — min: 0.011  median: 0.019  max: 0.976


## WBF merge (alternative to NMS)

In [7]:
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image
from collections import defaultdict

WBF_IOU_THR  = 0.25
WBF_SKIP_THR = 0.0001

# Image size lookup — WBF needs coords normalised to [0, 1]
print("Reading image sizes...")
img_sizes = {}
for p in TEST_IMG_DIR.iterdir():
    with Image.open(p) as im:
        img_sizes[int(p.stem)] = im.size  # (width, height)
print(f"{len(img_sizes)} images. Sample size: {next(iter(img_sizes.values()))}")

# Per-bag per-image predictions: bag_preds[bag_i][image_id] = [(x1,y1,x2,y2,score,cls), ...]
bag_preds = [defaultdict(list) for _ in range(N_BAGS)]

for i, pt in enumerate(BAG_WEIGHTS):
    print(f"bag_{i} inference...")
    model = YOLO(str(pt))
    for r in model.predict(
        source=str(TEST_IMG_DIR), conf=CONF_THR, imgsz=640,
        device=0, save=False, verbose=False, stream=True,
    ):
        image_id = int(Path(r.path).stem)
        xyxy  = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clses = r.boxes.cls.cpu().numpy().astype(int)
        for box, score, cls in zip(xyxy, confs, clses):
            bag_preds[i][image_id].append((*box.tolist(), float(score), int(cls)))
    del model
    torch.cuda.empty_cache()

print("Inference done. Running WBF...")

coco_preds_wbf = []

for image_id in img_sizes:
    w, h = img_sizes[image_id]
    boxes_list  = []
    scores_list = []
    labels_list = []

    for i in range(N_BAGS):
        dets = bag_preds[i][image_id]
        if dets:
            boxes_list.append([[d[0]/w, d[1]/h, d[2]/w, d[3]/h] for d in dets])
            scores_list.append([d[4] for d in dets])
            labels_list.append([d[5] for d in dets])
        else:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])

    if not any(scores_list):
        continue

    boxes_out, scores_out, labels_out = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        iou_thr=WBF_IOU_THR,
        skip_box_thr=WBF_SKIP_THR,
        conf_type="avg",
    )

    for box, score, cls in zip(boxes_out, scores_out, labels_out):
        x1, y1, x2, y2 = box[0]*w, box[1]*h, box[2]*w, box[3]*h
        coco_preds_wbf.append({
            "image_id":    image_id,
            "category_id": int(cls) + 1,
            "bbox":  [float(x1), float(y1), float(x2-x1), float(y2-y1)],
            "score": float(score),
        })

print(f"WBF detections: {len(coco_preds_wbf):,}")

Reading image sizes...
786 images. Sample size: (526, 341)
bag_0 inference...
bag_1 inference...
bag_2 inference...
bag_3 inference...
bag_4 inference...
bag_5 inference...
bag_6 inference...
Inference done. Running WBF...


c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:54: UserWarning: Y1 < 0 in box. Set it to 0.
  warnings.warn('Y1 < 0 in box. Set it to 0.')
c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:63: UserWarning: Y2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('Y2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


WBF detections: 199,212


In [8]:
OUT_JSON_WBF = PROJECT_ROOT / "bag_wbf_ensemble_preds.json"
OUT_JSON_WBF.write_text(json.dumps(coco_preds_wbf, indent=2))

scores_wbf = [d["score"] for d in coco_preds_wbf]
print(f"Written: {OUT_JSON_WBF}")
print(f"Total detections: {len(coco_preds_wbf):,}")
print(f"Score — min: {min(scores_wbf):.3f}  median: {float(np.median(scores_wbf)):.3f}  max: {max(scores_wbf):.3f}")

Written: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\bag_wbf_ensemble_preds.json
Total detections: 199,212
Score — min: 0.002  median: 0.007  max: 0.852
